# Decision Tree ID3

In [24]:
import numpy as np
import pandas as pd

https://www.geeksforgeeks.org/machine-learning/iterative-dichotomiser-3-id3-algorithm-from-scratch/

$$ \text{Entropia} = H(S) = - \sum_{i=1}^{c} p_i \log_{2}(p_i)$$

In [25]:
def entropia(coluna_alvo):
    elementos, contagem = np.unique(coluna_alvo, return_counts=True)

    valor_de_entropia = 0
    total = np.sum(contagem)


    for i in range (len(elementos)):
        p_i = contagem[i] / total

        log_2 = np.log2(p_i)

        valor_de_entropia += p_i * log_2

    return -valor_de_entropia

$$ \text{Ganho de informação} = IG (S, A) = H(S) - \sum_{v \in \text{Values(A)}} \frac{|S_v|}{|S|} H(S_v)$$

In [26]:
# S = conjunto de exemplos no nó atual
# A = atributo candidato a divisão
def ganho_de_informacao(S, A, alvo):
    
    # S[-1] -> coluna alvo
    h_s_atual = entropia(S[alvo])

    elementos, contagem = np.unique(S[A], return_counts=True)

    valor_de_entropia_atributo_A = 0
    total = np.sum(contagem)


    for i in range (len(elementos)):
        p_i = contagem[i] / total

        # calcula as sub-entropias de cada elemento do subconjunto S
        H_sv = entropia (S[S[alvo] == elementos[i]][alvo])

        valor_de_entropia_atributo_A += p_i * H_sv

    return h_s_atual - valor_de_entropia_atributo_A

In [27]:
def id3(dados, dados_originais, features, alvo, classe_do_no_pai = None):
    
    """Critérios de parada"""

    # se os dados estiverem puros
    if len(np.unique(dados[alvo])) == 1:
        return np.unique(dados[alvo])[0]
    
    # subconjunto vazio
    if (len(dados) == 0):

        # retorna a classe que mais aparece no dataset original (palpite para à folha)
        return np.unique(dados_originais[alvo])[np.argmax(
            np.unique(dados_originais[alvo], return_counts=True)[1])] 

    # atribui a classe majoritária quando não restam atributos
    if len(features) == 0:
        return classe_do_no_pai


    """Algoritmo"""

    # encontra a classe mais frequente
    classe_do_no_pai = np.unique(dados[alvo])[np.argmax(
        np.unique(dados[alvo], return_counts=True)[1])]

    max_ganho = 0
    melhor_feature = 0

    # encontrando a melhor feature -> maior ganho de informação
    for f in features:
        ganho_de_info = ganho_de_informacao(dados, f, alvo)

        if ganho_de_info > max_ganho:
            max_ganho = ganho_de_info
            melhor_feature = f


    decision_tree = {melhor_feature : {}}


    # remove o atributo para que não seja reutilizado nas decisões
    features = [f for f in features if f != melhor_feature]


    for elemento in np.unique(dados[melhor_feature]):

        # juntando os elementos iguais (considerando a melhor feature)
        sub_conjunto = dados[dados[melhor_feature] == elemento]

        # montando recursivamente as sub-árvores
        sub_arvore = id3(sub_conjunto, dados_originais, features, alvo, classe_do_no_pai)

        # alocando as sub-árvores na decision-tree (árvore original)
        decision_tree[melhor_feature][elemento] = sub_arvore

    return decision_tree

In [28]:
dados = [
    ["Sol", "Alta", "Alta", "Fraco", "Não"],
    ["Sol", "Alta", "Alta", "Forte", "Não"],
    ["Nublado", "Alta", "Alta", "Fraco", "Sim"],
    ["Chuva", "Média", "Alta", "Fraco", "Sim"],
    ["Chuva", "Baixa", "Normal", "Fraco", "Sim"],
    ["Chuva", "Baixa", "Normal", "Forte", "Não"],
    ["Nublado", "Baixa", "Normal", "Forte", "Sim"],
    ["Sol", "Média", "Alta", "Fraco", "Não"],
    ["Sol", "Baixa", "Normal", "Fraco", "Sim"],
    [ "Chuva", "Média", "Normal", "Fraco", "Sim"],
    [ "Sol", "Média", "Normal", "Forte", "Sim"],
    [ "Nublado", "Média", "Alta", "Forte", "Sim"],
    [ "Nublado", "Alta", "Normal", "Fraco", "Sim"],
    [ "Chuva", "Média", "Alta", "Forte", "Não"]
]

colunas = ["Tempo", "Temperatura", "Umidade", "Vento", "Jogar"]
df = pd.DataFrame(dados, columns=colunas)

In [29]:
decision_tree = id3(df, df, colunas, "Jogar")

print(decision_tree)

{'Tempo': {'Chuva': {'Temperatura': {'Baixa': {'Umidade': {'Normal': {'Vento': {'Forte': 'Não', 'Fraco': 'Sim'}}}}, 'Média': {'Umidade': {'Alta': {'Vento': {'Forte': 'Não', 'Fraco': 'Sim'}}, 'Normal': 'Sim'}}}}, 'Nublado': 'Sim', 'Sol': {'Temperatura': {'Alta': 'Não', 'Baixa': 'Sim', 'Média': {'Umidade': {'Alta': 'Não', 'Normal': 'Sim'}}}}}}


In [ ]:
# visualização feita com IA
def print_arvore(arvore, indent=""):
    if not isinstance(arvore, dict):
        print(f" -> {arvore}")
        return

    for atributo, ramos in arvore.items():
        print(f"[{atributo}]")
        itens = list(ramos.items())
        for i, (valor, sub_arvore) in enumerate(itens):
            ultimo = (i == len(itens) - 1)
            prefixo = "└── " if ultimo else "├── "
            extensao = "    " if ultimo else "│   "
            
            print(f"{indent}{prefixo}({valor})", end="")
            if isinstance(sub_arvore, dict):
                print()
                print_arvore(sub_arvore, indent + extensao)
            else:
                print(f" => {sub_arvore}")

print_arvore(decision_tree)

[Tempo]
├── (Chuva)
[Temperatura]
│   ├── (Baixa)
[Umidade]
│   │   └── (Normal)
[Vento]
│   │       ├── (Forte) => Não
│   │       └── (Fraco) => Sim
│   └── (Média)
[Umidade]
│       ├── (Alta)
[Vento]
│       │   ├── (Forte) => Não
│       │   └── (Fraco) => Sim
│       └── (Normal) => Sim
├── (Nublado) => Sim
└── (Sol)
[Temperatura]
    ├── (Alta) => Não
    ├── (Baixa) => Sim
    └── (Média)
[Umidade]
        ├── (Alta) => Não
        └── (Normal) => Sim
